In [22]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =============================================================================
# CORE FUNCTIONS
# =============================================================================

def compute_imp_probability(d, sigma1, sigma2, L):
    """Compute IMP analytical formula probability (pofr only)."""
    def sphere_cap(r1, r2, d):
        sc = 0.0
        if d <= max(r1, r2) - min(r1, r2):
            sc = min(4.0 / 3 * pi * r1 * r1 * r1,
                     4.0 / 3 * pi * r2 * r2 * r2)
        elif d >= r1 + r2:
            sc = 0
        else:
            sc = (pi / 12 / d * (r1 + r2 - d) * (r1 + r2 - d)) * \
                 (d * d + 2 * d * r1 - 3 * r1 * r1 + 2 * d * r2 + 6 * r1 * r2 -
                  3 * r2 * r2)
        return sc

    dist = max(d, 0.001)
    sigmai = sigma1
    sigmaj = sigma2

    voli = 4.0 / 3.0 * pi * sigmai * sigmai * sigmai
    volj = 4.0 / 3.0 * pi * sigmaj * sigmaj * sigmaj

    if dist < sigmai + sigmaj:
        xlvol = 4.0 / 3.0 * pi * (L / 2) * (L / 2) * (L / 2)
        fi = min(voli, xlvol)
        fj = min(volj, xlvol)
    else:
        di = dist - sigmaj - L / 2
        dj = dist - sigmai - L / 2
        fi = sphere_cap(sigmai, L / 2, abs(di))
        fj = sphere_cap(sigmaj, L / 2, abs(dj))

    pofr = fi * fj / voli / volj
    return pofr


def compute_mc_probability(p1, p2, sigma1, sigma2, L, N=100000):
    """Compute Monte Carlo probability by sampling uniformly within spheres."""
    q1_samples = sample_uniform_sphere(p1, sigma1, N)
    q2_samples = sample_uniform_sphere(p2, sigma2, N)
    distances = np.linalg.norm(q1_samples - q2_samples, axis=1)
    prob_mc = np.mean(distances < L)
    return prob_mc


def sample_uniform_sphere(center, radius, N):
    """Sample points uniformly within a sphere."""
    u = np.random.uniform(0, 1, N)
    r = radius * u**(1/3)
    theta = np.random.uniform(0, 2*np.pi, N)
    phi = np.arccos(2*np.random.uniform(0, 1, N) - 1)
    x = r * np.sin(phi) * np.cos(theta)
    y = r * np.sin(phi) * np.sin(theta)
    z = r * np.cos(phi)
    samples = np.column_stack((x, y, z)) + center
    return samples


In [23]:
# =============================================================================
# CELL 1: Fixed σ₁ = σ₂, varying distance (3x3 grid)
# =============================================================================

np.random.seed(42)

# Control variables
SAVE_PLOTS = False
L = 21.0
N_MC = 50000

print("="*70)
print("CELL 1: Distance vs Probability (σ₁ = σ₂, symmetric)")
print("="*70)

sigma_values = [1, 3, 5, 10, 10.5, 11, 12, 15, 20]
distances = np.linspace(0.000000, 50, 50)
p1 = np.array([0.0, 0.0, 0.0])

# Create 3x3 subplot grid with increased spacing
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f'σ₁=σ₂={s}Å' for s in sigma_values],
    vertical_spacing=0.12,  # Increased from 0.08
    horizontal_spacing=0.10  # Increased from 0.08
)

for idx, sigma in enumerate(sigma_values):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    #print(f"Computing σ={sigma}Å...")
    
    prob_imp = []
    prob_mc = []
    
    for d in distances:
        p2 = np.array([d, 0.0, 0.0])
        prob_imp.append(compute_imp_probability(d, sigma, sigma, L))
        prob_mc.append(compute_mc_probability(p1, p2, sigma, sigma, L, N=N_MC))
    
    # Add IMP trace
    fig.add_trace(
        go.Scatter(x=distances, y=prob_imp, mode='lines', 
                  name='IMP', line=dict(color='blue', width=2),
                  showlegend=(idx==0)),
        row=row, col=col
    )
    
    # Add MC trace
    fig.add_trace(
        go.Scatter(x=distances, y=prob_mc, mode='lines', 
                  name='MC', line=dict(color='red', width=2, dash='dash'),
                  showlegend=(idx==0)),
        row=row, col=col
    )
    
    # Add vertical line at L
    fig.add_vline(x=L, line=dict(color='green', dash='dot', width=1),
                 row=row, col=col)

for i in range(1, 4):
    for j in range(1, 4):
        fig.update_xaxes(
            title_text="Distance (Å)",
            row=i,
            col=j,
            title_font=dict(size=20),
            tickfont=dict(size=18)
        )
        fig.update_yaxes(
            title_text="Probability",
            row=i,
            col=j,
            title_font=dict(size=20),
            tickfont=dict(size=18),
            range=[0, 1.05]
        )

fig.update_layout(
    height=1100,
    width=1500,
    title_text=f"Distance vs Probability (σ₁=σ₂, L={L}Å)",
    title_font=dict(size=28),
    showlegend=True,
    legend=dict(font=dict(size=20)),
    font=dict(size=18),
    margin=dict(l=80, r=60, t=110, b=80)
)

for ann in fig.layout.annotations:
    ann.font = dict(size=18, color="black")
    ann.text = f"<b>{ann.text}</b>"

fig.show()

if SAVE_PLOTS:
    os.makedirs("output_figures", exist_ok=True)
    fig.write_html("output_figures/cell1_symmetric_sigma_grid.html")
    print(" Saved: output_figures/cell1_symmetric_sigma_grid.html")

print("="*70)
print("CELL 1 COMPLETE")
print("="*70)

CELL 1: Distance vs Probability (σ₁ = σ₂, symmetric)


CELL 1 COMPLETE


In [24]:
# =============================================================================
# CELL 2: Fixed σ₁=2.0, varying σ₂ (coarse-graining), distance sweep
# =============================================================================

np.random.seed(42)

# Control variables
SAVE_PLOTS = False
L = 21.0
N_MC = 50000
SIGMA1 = 2.0

print("\n" + "="*70)
print(f"CELL 2: Distance vs Probability (σ₁={SIGMA1}Å fixed, σ₂ varying)")
print("="*70)

sigma2_values = [1.0, 2, 5, 8, 10, 10.5, 11, 12, 20]
distances = np.linspace(0.5, 50, 50)
p1 = np.array([0.0, 0.0, 0.0])

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f'σ₁={SIGMA1}Å, σ₂={s2}Å' for s2 in sigma2_values],
    vertical_spacing=0.12,  # Increased spacing
    horizontal_spacing=0.10
)

for idx, sigma2 in enumerate(sigma2_values):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    #print(f"Computing σ₂={sigma2}Å...")
    
    prob_imp = []
    prob_mc = []
    
    for d in distances:
        p2 = np.array([d, 0.0, 0.0])
        prob_imp.append(compute_imp_probability(d, SIGMA1, sigma2, L))
        prob_mc.append(compute_mc_probability(p1, p2, SIGMA1, sigma2, L, N=N_MC))
    
    fig.add_trace(
        go.Scatter(x=distances, y=prob_imp, mode='lines',
                  name='IMP', line=dict(color='blue', width=2),
                  showlegend=(idx==0)),
        row=row, col=col
    )
    
    fig.add_trace(
        go.Scatter(x=distances, y=prob_mc, mode='lines',
                  name='MC', line=dict(color='red', width=2, dash='dash'),
                  showlegend=(idx==0)),
        row=row, col=col
    )
    
    fig.add_vline(x=L, line=dict(color='green', dash='dot', width=1),
                 row=row, col=col)

for i in range(1, 4):
    for j in range(1, 4):
        fig.update_xaxes(
            title_text="Distance (Å)",
            row=i,
            col=j,
            title_font=dict(size=20),
            tickfont=dict(size=18)
        )
        fig.update_yaxes(
            title_text="Probability",
            row=i,
            col=j,
            title_font=dict(size=20),
            tickfont=dict(size=18),
            range=[0, 1.05]
        )

fig.update_layout(
    height=1100,
    width=1500,
    title_text=f"Distance vs Probability (σ₁={SIGMA1}Å fixed, σ₂ varying, L={L}Å)",
    title_font=dict(size=28),
    showlegend=True,
    legend=dict(font=dict(size=20)),
    font=dict(size=18),
    margin=dict(l=80, r=60, t=110, b=80)
)

for ann in fig.layout.annotations:
    ann.font = dict(size=18, color="black")
    ann.text = f"<b>{ann.text}</b>"

fig.show()

if SAVE_PLOTS:
    os.makedirs("output_figures", exist_ok=True)
    fig.write_html("output_figures/cell2_asymmetric_coarsegraining.html")
    print("Saved: output_figures/cell2_asymmetric_coarsegraining.html")

print("="*70)
print("CELL 2 COMPLETE")
print("="*70)


CELL 2: Distance vs Probability (σ₁=2.0Å fixed, σ₂ varying)


CELL 2 COMPLETE


In [25]:
# =============================================================================
# CELL 3: Fixed distance, σ₁=2.0Å, varying σ₂
# =============================================================================

np.random.seed(42)

# Control variables
SAVE_PLOTS = False
L = 21.0
N_MC = 50000
SIGMA1 = 2.0

print("\n" + "="*70)
print(f"CELL 3: σ₂ vs Probability (σ₁={SIGMA1}Å, distance fixed)")
print("="*70)

distance_values = [5, 10, 15, 20, 21, 25, 30, 35, 40]
sigma2_range = np.linspace(0.5, 30, 50)
p1 = np.array([0.0, 0.0, 0.0])

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f'd={d}Å' for d in distance_values],
    vertical_spacing=0.12,  # Increased spacing
    horizontal_spacing=0.10
)

for idx, dist in enumerate(distance_values):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    #print(f"Computing d={dist}Å...")
    
    p2 = np.array([dist, 0.0, 0.0])
    prob_imp = []
    prob_mc = []
    
    for sigma2 in sigma2_range:
        prob_imp.append(compute_imp_probability(dist, SIGMA1, sigma2, L))
        prob_mc.append(compute_mc_probability(p1, p2, SIGMA1, sigma2, L, N=N_MC))
    
    fig.add_trace(
        go.Scatter(x=sigma2_range, y=prob_imp, mode='lines',
                  name='IMP', line=dict(color='blue', width=2),
                  showlegend=(idx==0)),
        row=row, col=col
    )
    
    fig.add_trace(
        go.Scatter(x=sigma2_range, y=prob_mc, mode='lines',
                  name='MC', line=dict(color='red', width=2, dash='dash'),
                  showlegend=(idx==0)),
        row=row, col=col
    )
    
    # Mark the regime boundary (σ₁ + σ₂ = d)
    boundary_sigma2 = dist - SIGMA1
    if 0.5 <= boundary_sigma2 <= 30:
        fig.add_vline(x=boundary_sigma2, line=dict(color='green', dash='dot', width=1),
                     row=row, col=col)

for i in range(1, 4):
    for j in range(1, 4):
        fig.update_xaxes(
            title_text="σ₂ (Å)",
            row=i,
            col=j,
            title_font=dict(size=20),
            tickfont=dict(size=18)
        )
        fig.update_yaxes(
            title_text="Probability",
            row=i,
            col=j,
            title_font=dict(size=20),
            tickfont=dict(size=18),
            range=[0, 1.05]
        )

fig.update_layout(
    height=1100,
    width=1500,
    title_text=f"σ₂ vs Probability (σ₁={SIGMA1}Å, distances fixed, L={L}Å)",
    title_font=dict(size=28),
    showlegend=True,
    legend=dict(font=dict(size=20)),
    font=dict(size=18),
    margin=dict(l=80, r=60, t=110, b=80)
)

for ann in fig.layout.annotations:
    ann.font = dict(size=18, color="black")
    ann.text = f"<b>{ann.text}</b>"

fig.show()

if SAVE_PLOTS:
    os.makedirs("output_figures", exist_ok=True)
    fig.write_html("output_figures/cell3_fixed_distance_vary_sigma2.html")
    print("✅ Saved: output_figures/cell3_fixed_distance_vary_sigma2.html")

print("="*70)
print("CELL 3 COMPLETE")
print("="*70)


CELL 3: σ₂ vs Probability (σ₁=2.0Å, distance fixed)


CELL 3 COMPLETE
